
# 🚀 Crypto Momentum Scanner + Ranked Asset Allocation v2

A research framework for a **live crypto scanner + ranked portfolio allocator**.

## What this version does

Instead of keeping a fixed 5-token universe, the system:

**Crypto Top 100 → Liquidity Filter → Momentum Scanner → Factor Scoring → Ranking → Portfolio Selection → Rebalance → Live Tracker**

The model combines:

- Momentum
- Relative strength
- Volatility
- Correlation
- Entropy / directional randomness
- Liquidity
- Volume expansion
- Trend confirmation

### Core idea

The **Top 100 universe is dynamic**, while the portfolio is concentrated in the strongest current opportunities.

The notebook is intentionally designed as a **research engine first**. It does not place live orders.

> Educational/research use only. Crypto is highly volatile and the strategy can lose substantial capital.



# Architecture

```text
                    ┌──────────────────┐
                    │ Crypto Universe   │
                    │ Top 100 by MCap   │
                    └────────┬─────────┘
                             ↓
                    ┌──────────────────┐
                    │ Liquidity Filter │
                    │ Volume / Spread  │
                    └────────┬─────────┘
                             ↓
                    ┌──────────────────┐
                    │ Momentum Scanner │
                    │ 1D / 3D / 7D / 30D
                    └────────┬─────────┘
                             ↓
          ┌──────────────────┼──────────────────┐
          ↓                  ↓                  ↓
     Volatility          Correlation         Entropy
          └──────────────────┼──────────────────┘
                             ↓
                    ┌──────────────────┐
                    │ Composite Score  │
                    │ Equal-weighted   │
                    └────────┬─────────┘
                             ↓
                    ┌──────────────────┐
                    │ Ranking Top 10   │
                    └────────┬─────────┘
                             ↓
                    ┌──────────────────┐
                    │ Portfolio Top N  │
                    └────────┬─────────┘
                             ↓
                    ┌──────────────────┐
                    │ Rebalance Engine │
                    └────────┬─────────┘
                             ↓
                    ┌──────────────────┐
                    │ Live Tracker     │
                    └──────────────────┘
```

The scanner and portfolio allocator are separate modules so the same scoring engine can later feed:

- Streamlit
- FastAPI
- Telegram/Discord alerts
- TradingView webhooks
- exchange execution


In [ ]:

# Install once if required
# %pip install pandas numpy requests ccxt matplotlib scipy

import time
import warnings
warnings.filterwarnings("ignore")

import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime, timezone

import ccxt


In [ ]:

# ============================================================
# CONFIGURATION
# ============================================================

CONFIG = {
    # Universe
    "top_market_cap": 100,
    "quote_currency": "USDT",

    # Liquidity
    "min_24h_volume_usd": 5_000_000,

    # Momentum windows
    "momentum_windows": [1, 3, 7, 30],
    "momentum_weights": [0.10, 0.20, 0.30, 0.40],

    # Technical confirmation
    "ema_fast": 20,
    "ema_slow": 50,

    # Risk / noise
    "volatility_window": 30,
    "correlation_window": 60,
    "entropy_window": 60,

    # Volume
    "volume_window": 20,

    # Portfolio
    "portfolio_size": 5,
    "rebalance_frequency": "daily",

    # Factor weights
    "factor_weights": {
        "momentum": 0.25,
        "volatility": 0.20,
        "correlation": 0.15,
        "entropy": 0.15,
        "volume": 0.15,
        "trend": 0.10,
    },

    # Position constraints
    "max_position_weight": 0.30,
    "min_score": 0.55,

    # Trading assumptions
    "transaction_cost": 0.001,
    "slippage": 0.0005,

    # Live scanner
    "refresh_seconds": 60,
}

print(CONFIG)


In [ ]:

# ============================================================
# DATA PROVIDERS
# ============================================================

COINGECKO_URL = "https://api.coingecko.com/api/v3/coins/markets"

def get_top_100_coins():
    params = {
        "vs_currency": "usd",
        "order": "market_cap_desc",
        "per_page": CONFIG["top_market_cap"],
        "page": 1,
        "sparkline": "false",
    }

    r = requests.get(COINGECKO_URL, params=params, timeout=20)
    r.raise_for_status()

    df = pd.DataFrame(r.json())

    cols = [
        "id", "symbol", "name", "market_cap_rank",
        "current_price", "market_cap",
        "total_volume", "price_change_percentage_24h"
    ]

    return df[cols].copy()


top100 = get_top_100_coins()

display(top100.head(20))



## Important universe rule

Market-cap ranking and tradability are different concepts.

A token can be Top 100 by market cap but still have:

- poor liquidity
- large spreads
- limited exchange availability
- unstable pricing
- restricted trading pairs

Therefore the scanner applies a **minimum 24h volume filter** before technical ranking.


In [ ]:

# ============================================================
# EXCHANGE CONNECTION
# ============================================================

exchange = ccxt.binance({
    "enableRateLimit": True,
})

markets = exchange.load_markets()

usdt_symbols = {
    m["base"].upper(): symbol
    for symbol, m in markets.items()
    if m.get("spot")
    and m.get("quote") == CONFIG["quote_currency"]
    and m.get("active", True)
}

top100["BASE"] = top100["symbol"].str.upper()

universe = top100[
    (top100["total_volume"] >= CONFIG["min_24h_volume_usd"])
    & (top100["BASE"].isin(usdt_symbols))
].copy()

universe["symbol"] = universe["BASE"].map(usdt_symbols)

print(f"Top-100 assets: {len(top100)}")
print(f"Tradable + liquid assets: {len(universe)}")

display(universe[[
    "market_cap_rank", "name", "symbol",
    "market_cap", "total_volume"
]].head(30))


In [ ]:

# ============================================================
# OHLCV DOWNLOAD
# ============================================================

def fetch_ohlcv(symbol, timeframe="1d", limit=120):
    try:
        raw = exchange.fetch_ohlcv(
            symbol,
            timeframe=timeframe,
            limit=limit
        )

        df = pd.DataFrame(
            raw,
            columns=["timestamp", "open", "high", "low", "close", "volume"]
        )

        df["timestamp"] = pd.to_datetime(df["timestamp"], unit="ms", utc=True)
        df = df.set_index("timestamp")

        return df

    except Exception as e:
        print("Data error:", symbol, str(e)[:120])
        return None


price_data = {}

for symbol in universe["symbol"].head(CONFIG["top_market_cap"]):
    df = fetch_ohlcv(symbol)

    if df is not None and len(df) >= 70:
        price_data[symbol] = df

print("Assets with sufficient OHLCV:", len(price_data))


In [ ]:

# ============================================================
# FACTOR ENGINE
# ============================================================

def momentum_score_series(close):
    windows = CONFIG["momentum_windows"]
    weights = CONFIG["momentum_weights"]

    parts = []

    for w, weight in zip(windows, weights):
        parts.append(close.pct_change(w) * weight)

    return sum(parts)


def realized_volatility(close, window):
    return close.pct_change().rolling(window).std() * np.sqrt(365)


def binary_entropy(close, window):
    r = close.pct_change()
    positive_probability = (r > 0).rolling(window).mean()

    p = positive_probability.clip(1e-12, 1 - 1e-12)

    return -(
        p * np.log2(p)
        + (1 - p) * np.log2(1 - p)
    )


def volume_ratio(volume, window):
    return volume / volume.rolling(window).mean()


def trend_score(close):
    fast = close.ewm(span=CONFIG["ema_fast"], adjust=False).mean()
    slow = close.ewm(span=CONFIG["ema_slow"], adjust=False).mean()

    return (fast / slow - 1)


def average_pairwise_correlation(asset_symbol, returns_matrix, window):
    window_data = returns_matrix.tail(window)

    if asset_symbol not in window_data.columns:
        return np.nan

    corr = window_data.corr()[asset_symbol].drop(asset_symbol, errors="ignore")

    return corr.abs().mean()


In [ ]:

# ============================================================
# BUILD RAW FACTOR SNAPSHOT
# ============================================================

def build_factor_snapshot(price_data):
    rows = []

    close_matrix = pd.DataFrame({
        symbol: df["close"]
        for symbol, df in price_data.items()
    })

    returns_matrix = close_matrix.pct_change()

    for symbol, df in price_data.items():

        close = df["close"]
        volume = df["volume"]

        if len(close) < 65:
            continue

        row = {
            "symbol": symbol,
            "price": close.iloc[-1],

            "momentum": momentum_score_series(close).iloc[-1],

            "volatility": realized_volatility(
                close,
                CONFIG["volatility_window"]
            ).iloc[-1],

            "entropy": binary_entropy(
                close,
                CONFIG["entropy_window"]
            ).iloc[-1],

            "volume_ratio": volume_ratio(
                volume,
                CONFIG["volume_window"]
            ).iloc[-1],

            "trend": trend_score(close).iloc[-1],

            "correlation": average_pairwise_correlation(
                symbol,
                returns_matrix,
                CONFIG["correlation_window"]
            ),
        }

        rows.append(row)

    return pd.DataFrame(rows)


raw = build_factor_snapshot(price_data)

display(raw.sort_values("momentum", ascending=False).head(20))


In [ ]:

# ============================================================
# CROSS-SECTIONAL SCORING
# ============================================================

def percentile_rank(series, higher_is_better=True):
    r = series.rank(pct=True, method="average")

    if higher_is_better:
        return r

    return 1 - r


def build_scores(raw):
    df = raw.copy()

    df["momentum_score"] = percentile_rank(
        df["momentum"], True
    )

    df["volatility_score"] = percentile_rank(
        df["volatility"], False
    )

    df["correlation_score"] = percentile_rank(
        df["correlation"], False
    )

    df["entropy_score"] = percentile_rank(
        df["entropy"], False
    )

    df["volume_score"] = percentile_rank(
        df["volume_ratio"], True
    )

    df["trend_score"] = percentile_rank(
        df["trend"], True
    )

    fw = CONFIG["factor_weights"]

    df["composite_score"] = (
        fw["momentum"] * df["momentum_score"]
        + fw["volatility"] * df["volatility_score"]
        + fw["correlation"] * df["correlation_score"]
        + fw["entropy"] * df["entropy_score"]
        + fw["volume"] * df["volume_score"]
        + fw["trend"] * df["trend_score"]
    )

    df["rank"] = (
        df["composite_score"]
        .rank(ascending=False, method="min")
        .astype(int)
    )

    return df.sort_values("composite_score", ascending=False)


scores = build_scores(raw)

columns = [
    "rank", "symbol", "price",
    "momentum", "volatility", "correlation",
    "entropy", "volume_ratio", "trend",
    "composite_score"
]

display(scores[columns].head(20).round(4))



# 🔥 Momentum Scanner

The scanner deliberately separates:

### 1. Fast momentum
1D / 3D / 7D returns

Useful for detecting new acceleration.

### 2. Medium momentum
30D return

Useful for avoiding assets that are only experiencing a one-day spike.

### 3. Trend confirmation
EMA20 > EMA50

### 4. Volume confirmation
Current volume / 20-day average volume

### 5. Risk filters
Volatility, entropy and correlation.

This gives us a better definition of a **momentum trade** than simply sorting by 24h percentage change.


In [ ]:

# ============================================================
# MOMENTUM TRADE FILTER
# ============================================================

def momentum_candidates(scores):
    x = scores.copy()

    x["momentum_trade"] = (
        (x["momentum"] > 0)
        & (x["trend"] > 0)
        & (x["volume_ratio"] > 1.0)
        & (x["composite_score"] >= CONFIG["min_score"])
    )

    return x[x["momentum_trade"]].copy()


candidates = momentum_candidates(scores)

display(
    candidates[columns]
    .head(20)
    .round(4)
)


In [ ]:

# ============================================================
# PORTFOLIO ALLOCATOR
# ============================================================

def allocate_portfolio(scores):
    eligible = scores[
        (scores["composite_score"] >= CONFIG["min_score"])
        & (scores["momentum"] > 0)
        & (scores["trend"] > 0)
    ].copy()

    selected = eligible.head(CONFIG["portfolio_size"]).copy()

    if selected.empty:
        return pd.DataFrame(
            columns=["symbol", "rank", "score", "weight"]
        )

    # Score-weighted allocation
    raw_weights = selected["composite_score"] / selected["composite_score"].sum()

    raw_weights = raw_weights.clip(
        upper=CONFIG["max_position_weight"]
    )

    weights = raw_weights / raw_weights.sum()

    portfolio = pd.DataFrame({
        "symbol": selected["symbol"],
        "rank": selected["rank"],
        "score": selected["composite_score"],
        "weight": weights.values,
    })

    return portfolio


portfolio = allocate_portfolio(scores)

display(portfolio.round(4))



# 🔄 Rebalance Engine

The live system should **not rebalance every time the ranking changes by one position**.

A better production rule is to use **hysteresis**:

- Existing position stays unless its rank falls below an exit threshold.
- New asset enters only if its score exceeds the weakest holding by a meaningful margin.
- Rebalance only when weight drift or score deterioration is significant.

This reduces churn and transaction costs.

Example:

```text
Current holding: SOL rank #4
New candidate: DOGE rank #5

→ Do NOT automatically sell SOL.

New candidate: DOGE rank #1
SOL rank #12

→ Rebalance.
```


In [ ]:

# ============================================================
# LIVE MARKET TICKER
# ============================================================

def fetch_live_tickers(symbols):
    result = []

    for symbol in symbols:
        try:
            ticker = exchange.fetch_ticker(symbol)

            result.append({
                "symbol": symbol,
                "last": ticker.get("last"),
                "bid": ticker.get("bid"),
                "ask": ticker.get("ask"),
                "24h_change_pct": ticker.get("percentage"),
                "24h_volume": ticker.get("quoteVolume"),
                "timestamp": ticker.get("timestamp"),
            })

        except Exception as e:
            print("Ticker error:", symbol, str(e)[:80])

    return pd.DataFrame(result)


live = fetch_live_tickers(
    universe["symbol"].head(50).tolist()
)

display(live.head(20))


In [ ]:

# ============================================================
# PORTFOLIO LIVE TRACKER
# ============================================================

def portfolio_tracker(portfolio):
    if portfolio.empty:
        return pd.DataFrame()

    live = fetch_live_tickers(portfolio["symbol"].tolist())

    tracker = portfolio.merge(
        live,
        on="symbol",
        how="left"
    )

    tracker["weighted_24h_contribution"] = (
        tracker["weight"] * tracker["24h_change_pct"] / 100
    )

    return tracker.sort_values(
        "weight",
        ascending=False
    )


tracker = portfolio_tracker(portfolio)

display(tracker.round(4))


In [ ]:

# ============================================================
# SCANNER DASHBOARD
# ============================================================

def scanner_dashboard(scores, top_n=20):
    x = scores.head(top_n).copy()

    x["signal"] = np.where(
        x["momentum_trade"],
        "🔥 MOMENTUM",
        "WATCH"
    ) if "momentum_trade" in x.columns else "WATCH"

    display_cols = [
        "rank",
        "symbol",
        "composite_score",
        "momentum",
        "volume_ratio",
        "trend",
        "volatility",
        "correlation",
        "entropy",
        "signal"
    ]

    return x[display_cols].round(4)


dashboard = scanner_dashboard(
    momentum_candidates(scores),
    20
)

display(dashboard)



# 🧠 Signal interpretation

A high score does **not** necessarily mean "buy immediately".

Think of the ranking as a **capital allocation priority system**.

Example:

| Rank | Interpretation |
|---|---|
| 1–3 | strongest candidates |
| 4–10 | watch / potential entry |
| 11–25 | neutral |
| 26+ | low priority |

Then use the live scanner to detect **rank acceleration**:

```text
Yesterday      Today
BTC     #12 → #11
ETH     #8  → #7
SOL     #17 → #3   🔥
AVAX    #6  → #18  ⚠️
```

The change in rank can itself become a factor:

\[
RankMomentum_t = Rank_{t-1} - Rank_t
\]

A large positive value means the asset is climbing rapidly through the cross-sectional universe.


In [ ]:

# ============================================================
# RANK MOMENTUM / ACCELERATION
# ============================================================

# For production:
# save every scanner snapshot to a database.
#
# Example schema:
#
# scanner_history:
# timestamp
# symbol
# rank
# composite_score
# momentum
# volume_ratio
# volatility
# correlation
# entropy
#
# Then:

def calculate_rank_change(current, previous):
    x = current[["symbol", "rank", "composite_score"]].copy()

    p = previous[["symbol", "rank"]].rename(
        columns={"rank": "previous_rank"}
    )

    x = x.merge(p, on="symbol", how="left")

    x["rank_change"] = (
        x["previous_rank"] - x["rank"]
    )

    return x.sort_values(
        "rank_change",
        ascending=False
    )


print("Rank acceleration requires a previous saved scanner snapshot.")



# 📊 Research: Factor Ablation

This is one of the most important parts of the project.

Instead of saying:

> "Entropy is useful."

test:

```text
Model A = Momentum
Model B = Momentum + Volatility
Model C = Momentum + Volatility + Correlation
Model D = Momentum + Volatility + Correlation + Entropy
Model E = All factors
```

If Model D does not improve out-of-sample performance, entropy should probably be removed or redesigned.

The same logic applies to correlation and volume.


In [ ]:

# ============================================================
# CURRENT FACTOR CORRELATION
# ============================================================

factor_cols = [
    "momentum_score",
    "volatility_score",
    "correlation_score",
    "entropy_score",
    "volume_score",
    "trend_score",
]

factor_corr = scores[factor_cols].corr()

display(factor_corr.round(3))

plt.figure(figsize=(9, 7))
plt.imshow(factor_corr, aspect="auto")
plt.xticks(range(len(factor_cols)), factor_cols, rotation=45, ha="right")
plt.yticks(range(len(factor_cols)), factor_cols)
plt.colorbar(label="Correlation")
plt.title("Cross-Sectional Factor Score Correlation")
plt.tight_layout()
plt.show()



# ⚙️ Production roadmap

The notebook is now the **research prototype**. The next architecture should split it into services.

### `scanner/`

```text
universe.py
market_data.py
liquidity.py
momentum.py
volatility.py
correlation.py
entropy.py
ranking.py
signals.py
```

### `portfolio/`

```text
allocator.py
rebalance.py
risk.py
position_sizing.py
drawdown.py
```

### `live/`

```text
websocket.py
ticker_tracker.py
rank_tracker.py
alert_engine.py
```

### `storage/`

Use:

- PostgreSQL / Supabase
- TimescaleDB if tick/time-series volume becomes large
- Redis for current live state

Store every scanner snapshot so you can calculate:

- rank acceleration
- score acceleration
- factor deterioration
- signal persistence
- turnover
- realized alpha after signal

### `dashboard/`

A Streamlit dashboard can show:

```text
┌─────────────────────────────────────────────────────┐
│ CRYPTO RANKED ALLOCATION                            │
├─────────────────────────────────────────────────────┤
│ Portfolio: BTC 25% | SOL 24% | ETH 21% | ...       │
├─────────────────────────────────────────────────────┤
│ 🔥 MOMENTUM BREAKOUTS                               │
│ SOL   #3   +42 rank acceleration                   │
│ DOGE  #7   +18                                        │
│ SUI   #9   +14                                        │
├─────────────────────────────────────────────────────┤
│ FACTOR MATRIX                                       │
│ Momentum | Vol | Corr | Entropy | Volume | Trend   │
├─────────────────────────────────────────────────────┤
│ REBALANCE                                           │
│ HOLD  → SOL                                          │
│ ADD   → SUI                                          │
│ REDUCE→ XRP                                          │
└─────────────────────────────────────────────────────┘
```

## The key evolution

The strongest version of this idea is **not simply "Top 5 coins by score."**

It is:

**Top 100 → liquid universe → detect momentum acceleration → score quality of momentum → rank → allocate → continuously monitor → rebalance only when the signal meaningfully changes.**

That gives you a proper **cross-sectional crypto momentum / relative-strength engine** rather than a static portfolio.


In [ ]:

# ============================================================
# OPTIONAL: CONTINUOUS LIVE SCANNER
# ============================================================
#
# Uncomment to run a simple terminal-style scanner.
# For production, replace this with WebSockets + database
# persistence rather than repeatedly polling everything.

def run_live_scanner():
    while True:
        try:
            print("\n" + "=" * 80)
            print("LIVE SCAN:", datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC"))
            print("=" * 80)

            top100_live = get_top_100_coins()

            universe_live = top100_live[
                top100_live["total_volume"] >= CONFIG["min_24h_volume_usd"]
            ].copy()

            universe_live["BASE"] = universe_live["symbol"].str.upper()
            universe_live = universe_live[
                universe_live["BASE"].isin(usdt_symbols)
            ]

            # Re-fetch OHLCV for the filtered universe.
            live_data = {}

            for symbol in universe_live["BASE"].map(usdt_symbols).head(100):
                df = fetch_ohlcv(symbol)

                if df is not None and len(df) >= 70:
                    live_data[symbol] = df

            raw_live = build_factor_snapshot(live_data)
            scores_live = build_scores(raw_live)
            candidates_live = momentum_candidates(scores_live)
            portfolio_live = allocate_portfolio(scores_live)

            print("\nTOP MOMENTUM CANDIDATES")
            display(
                candidates_live[columns].head(15).round(4)
            )

            print("\nPORTFOLIO")
            display(
                portfolio_live.round(4)
            )

            time.sleep(CONFIG["refresh_seconds"])

        except KeyboardInterrupt:
            print("Scanner stopped.")
            break

        except Exception as e:
            print("Scanner error:", e)
            time.sleep(CONFIG["refresh_seconds"])

# run_live_scanner()
